In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Transforms
transform = transforms.Compose([
    transforms.Resize((28, 28)),          # Resize to 28x28
    transforms.Grayscale(3),              # Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(),                # Convert to Tensor
    transforms.Normalize(                 # Normalize with ImageNet stats
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load Dataset
train_dataset = EMNIST(
    root='./data',
    split='letters',
    train=True,
    download=True,
    transform=transform
)

test_dataset = EMNIST(
    root='./data',
    split='letters',
    train=False,
    download=True,
    transform=transform
)

# Create DataLoaders
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train loader ready")
print("Test loader ready")

# Display Sample Images
data_iter = iter(train_loader)
images, labels = next(data_iter)

letters = [chr(ord('A') + i) for i in range(26)]

fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img = images[i].permute(1, 2, 0).numpy()

    # For visualization only (undo normalization effect)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    ax.imshow(img)
    ax.set_title(letters[labels[i].item() - 1])  # labels are 1–26
    ax.axis("off")

plt.show()

print("Shape of one image:", images[0].shape)  # (3, 28, 28)

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load EfficientNetV2-Small (no pretrained weights to avoid error)
model = models.efficientnet_v2_s(weights=None)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head (26 classes)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 26)

# Train classifier only
for param in model.classifier.parameters():
    param.requires_grad = True

# Move model to device
model = model.to(device)

print("Model ready")

In [ ]:
# Write your code here
from tqdm import tqdm
import torch


# Note: EMNIST letters labels are 1-26, so we subtract 1 to make them 0-2

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images = images.to(device)
        labels = (labels - 1).to(device)  #  convert 1-26 -> 0-25

        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images = images.to(device)
            labels = (labels - 1).to(device)  # convert 1-26 -> 0-25

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Move model to device (model is from Part 2)
model = model.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 2  # Faster

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

# Plot Training & Validation Loss
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

# Plot Training & Validation Accuracy
plt.figure(figsize=(8, 4))
plt.plot(train_accuracies, label="Training Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()

In [ ]:
# Note: The code runs correctly and the results are valid,
# however the training process takes a long time (especially on CPU).

In [ ]:
# Write your code here

            # Validation loop with Test Time Augmentation (TTA)
def validate_tta(model, dataloader, criterion, device):
    model.eval()  # set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            # predictions on original images
            out_original = model(images)

            # predictions on horizontally flipped images
            h_flipped = torch.flip(images, dims=[3])
            out_h = model(h_flipped)

            # predictions on vertically flipped images
            v_flipped = torch.flip(images, dims=[2])
            out_v = model(v_flipped)

            # average predictions
            outputs = (out_original + out_h + out_v) / 3

            # compute loss
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # compute accuracy
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total

    return avg_loss, accuracy

